# Multi-Task Learning — Detection + Depth + Segmentation

## 목표
- 단일 ResNet backbone으로 3개 태스크 **동시 학습**
  - **Detection**: FCOS style anchor-free head
  - **Depth**: Monocular depth estimation head
  - **Segmentation**: Semantic segmentation head
- **GradNorm** (ICML 2018): 태스크별 gradient 크기 자동 균형
- Single vs Multi-task 성능 비교 (positive/negative transfer 분석)

## 핵심 논문
- **GradNorm: Gradient Normalization for Adaptive Loss Balancing** (Chen et al., ICML 2018)
- **Multi-Task Learning as Multi-Objective Optimization** (Sener et al., NeurIPS 2018)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Multi-Task Learning 이론

### 왜 Multi-Task Learning인가?

```
Single-Task:     Backbone → Det Head      (3개 모델 = 3x 연산)
                 Backbone → Depth Head
                 Backbone → Seg Head

Multi-Task:      Backbone → Det Head      (1개 backbone = 1x 연산)
                         ↗ Depth Head     + 태스크 간 공유 표현 학습
                         ↘ Seg Head
```

### 핵심 문제: Gradient 충돌
- 태스크마다 loss 크기가 다름 (예: Det loss=5.0, Depth loss=0.01)
- 단순 합산 시 큰 loss가 학습을 지배 → **negative transfer**

### GradNorm 해법
```
각 태스크 i의 gradient norm G_i를 측정
목표: 모든 태스크의 G_i가 동일하게 유지
손실 가중치 w_i를 자동으로 조정:
  - G_i > 평균 → w_i 감소 (이 태스크가 너무 강함)
  - G_i < 평균 → w_i 증가 (이 태스크가 너무 약함)
```

In [ ]:
# ────────────────────────────────────────────────
# 합성 멀티태스크 데이터셋
# (Detection bbox + Depth map + Seg mask 동시 생성)
# ────────────────────────────────────────────────

IMG_H, IMG_W = 256, 256
NUM_CLASSES  = 5   # 배경, 차량, 보행자, 도로, 하늘

CLASS_COLORS = {
    0: (40, 40, 40),    # 배경
    1: (0, 0, 200),     # 차량 (파랑)
    2: (200, 100, 0),   # 보행자 (주황)
    3: (80, 80, 80),    # 도로 (회색)
    4: (100, 180, 255), # 하늘 (하늘색)
}


def generate_scene(img_h=IMG_H, img_w=IMG_W, seed=None):
    """도시 장면 합성: 이미지 + 깊이 맵 + 세그멘테이션 마스크 + detection bbox 반환"""
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    img     = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    seg     = np.zeros((img_h, img_w), dtype=np.int64)      # class 인덱스
    depth   = np.ones((img_h, img_w), dtype=np.float32) * 50.0  # 기본 50m

    # ── 하늘 (상단 30%) ─────────────────────────
    sky_h = int(img_h * 0.30)
    img[:sky_h] = np.array(CLASS_COLORS[4])
    seg[:sky_h] = 4
    depth[:sky_h] = 100.0

    # ── 도로 (중간~하단) ────────────────────────
    road_start = int(img_h * 0.45)
    img[road_start:] = np.array(CLASS_COLORS[3]) + np.random.randint(-10, 10, (img_h-road_start, img_w, 3)).astype(np.int32).clip(0,255).astype(np.uint8)
    img[road_start:] = np.clip(img[road_start:].astype(int) + np.random.randint(-5,5,(img_h-road_start,img_w,3)), 0,255).astype(np.uint8)
    seg[road_start:] = 3

    # 도로 깊이: 가까울수록 깊이 작음 (원근)
    for row in range(road_start, img_h):
        t = (row - road_start) / (img_h - road_start + 1e-6)
        depth[row, :] = max(1.0, 30.0 * (1.0 - t) + 2.0)

    # ── 배경 (건물 등) ──────────────────────────
    for _ in range(random.randint(2, 4)):
        bx = random.randint(0, img_w-40)
        bh = random.randint(30, int(img_h*0.35))
        bw = random.randint(20, 60)
        by = random.randint(int(img_h*0.20), int(img_h*0.45))
        color = tuple(np.random.randint(60, 150, 3).tolist())
        img[by:by+bh, bx:bx+bw] = color
        seg[by:by+bh, bx:bx+bw] = 0
        depth[by:by+bh, bx:bx+bw] = random.uniform(15, 35)

    # ── 차량 (bbox 반환) ────────────────────────
    bboxes = []
    n_vehicles = random.randint(1, 4)
    for _ in range(n_vehicles):
        vw = random.randint(30, 60)
        vh = random.randint(20, 40)
        vx = random.randint(0, img_w - vw)
        vy = random.randint(road_start, img_h - vh)
        obj_depth = random.uniform(5, 25)
        color = tuple((np.array(CLASS_COLORS[1]) + np.random.randint(-20, 20, 3)).clip(0,255).tolist())
        img[vy:vy+vh, vx:vx+vw] = color
        seg[vy:vy+vh, vx:vx+vw] = 1
        depth[vy:vy+vh, vx:vx+vw] = obj_depth
        bboxes.append([vx/img_w, vy/img_h, (vx+vw)/img_w, (vy+vh)/img_h, 1])  # [x1,y1,x2,y2,cls]

    # ── 보행자 ──────────────────────────────────
    n_ped = random.randint(0, 2)
    for _ in range(n_ped):
        pw = random.randint(10, 20)
        ph = random.randint(25, 45)
        px = random.randint(0, img_w - pw)
        py = random.randint(road_start, img_h - ph)
        obj_depth = random.uniform(5, 20)
        color = tuple((np.array(CLASS_COLORS[2]) + np.random.randint(-20, 20, 3)).clip(0,255).tolist())
        img[py:py+ph, px:px+pw] = color
        seg[py:py+ph, px:px+pw] = 2
        depth[py:py+ph, px:px+pw] = obj_depth
        bboxes.append([px/img_w, py/img_h, (px+pw)/img_w, (py+ph)/img_h, 2])

    # 약간의 노이즈
    noise = np.random.randint(0, 15, img.shape, dtype=np.uint8)
    img = np.clip(img.astype(int) + noise - 7, 0, 255).astype(np.uint8)

    return img, depth, seg, bboxes


# 샘플 시각화
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for col in range(3):
    img, depth, seg, bboxes = generate_scene(seed=col*5)

    axes[0, col].imshow(img)
    axes[0, col].set_title(f'RGB (bbox {len(bboxes)}개)')
    for bbox in bboxes:
        x1,y1,x2,y2,cls = bbox
        rect = plt.Rectangle((x1*IMG_W, y1*IMG_H), (x2-x1)*IMG_W, (y2-y1)*IMG_H,
                              fill=False, edgecolor='red', linewidth=2)
        axes[0, col].add_patch(rect)

    im = axes[1, col].imshow(depth, cmap='plasma', vmin=0, vmax=50)
    axes[1, col].set_title('Depth Map (m)')
    plt.colorbar(im, ax=axes[1, col], fraction=0.046)

    color_seg = np.zeros((*seg.shape, 3), dtype=np.uint8)
    for cls_id, color in CLASS_COLORS.items():
        color_seg[seg == cls_id] = color
    axes[2, col].imshow(color_seg)
    axes[2, col].set_title('Seg Mask')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('합성 멀티태스크 데이터셋 (RGB / Depth / Segmentation)', fontsize=12)
plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
class MTLDataset(Dataset):
    """멀티태스크 학습 데이터셋 — Detection + Depth + Segmentation"""

    def __init__(self, size=1500, augment=True, seed_offset=0):
        self.size   = size
        self.augment = augment
        self.seed_offset = seed_offset
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        seed = idx + self.seed_offset
        img_np, depth_np, seg_np, bboxes = generate_scene(seed=seed)

        img_pil = Image.fromarray(img_np)
        if self.augment and random.random() > 0.5:
            img_pil = T.functional.adjust_brightness(img_pil, random.uniform(0.8, 1.2))

        img_t   = self.transform(img_pil)
        depth_t = torch.from_numpy(depth_np).unsqueeze(0)   # (1, H, W)
        seg_t   = torch.from_numpy(seg_np).long()            # (H, W)

        # Detection: (N, 5) bbox [x1,y1,x2,y2,cls] → 고정 크기 텐서로 패딩
        max_boxes = 8
        bbox_t = torch.zeros(max_boxes, 5)
        if bboxes:
            boxes = torch.tensor(bboxes[:max_boxes], dtype=torch.float32)
            bbox_t[:len(boxes)] = boxes

        return img_t, depth_t, seg_t, bbox_t


train_ds = MTLDataset(size=1500, augment=True,  seed_offset=0)
val_ds   = MTLDataset(size=300,  augment=False, seed_offset=100000)

train_loader = DataLoader(train_ds, batch_size=12, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=12, shuffle=False, num_workers=0, pin_memory=True)

imgs, depths, segs, bboxes = next(iter(train_loader))
print(f'이미지:    {imgs.shape}')
print(f'Depth:     {depths.shape}')
print(f'Seg:       {segs.shape}')
print(f'BBoxes:    {bboxes.shape}')

## 2. MTL 모델 구현

### 구조: 공유 backbone + 3개 태스크 헤드

```
ResNet-18 Backbone
    ├── layer1 (64ch, stride 4)
    ├── layer2 (128ch, stride 8)
    ├── layer3 (256ch, stride 16)
    └── layer4 (512ch, stride 32)
              ↓
    FPN (Feature Pyramid Network)
              ↓
    ┌─────────┬──────────┬──────────┐
    ↓         ↓          ↓
  Det Head  Depth Head  Seg Head
  (FCOS)   (U-Net up)  (DeepLab-lite)
```

In [ ]:
class FPNNeck(nn.Module):
    """Feature Pyramid Network — 다중 스케일 feature 융합"""

    def __init__(self, in_channels=(64,128,256,512), out_channels=128):
        super().__init__()
        # 각 레이어를 out_channels로 통일
        self.laterals = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1) for c in in_channels
        ])
        self.outputs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels
        ])

    def forward(self, features):
        # features: [f1, f2, f3, f4] from ResNet
        laterals = [l(f) for l, f in zip(self.laterals, features)]

        # Top-down pathway
        for i in range(len(laterals)-2, -1, -1):
            laterals[i] = laterals[i] + F.interpolate(
                laterals[i+1], size=laterals[i].shape[-2:], mode='nearest'
            )

        outputs = [conv(lat) for conv, lat in zip(self.outputs, laterals)]
        return outputs  # [p1, p2, p3, p4] 다중 스케일 feature


class DetectionHead(nn.Module):
    """FCOS-style anchor-free detection head"""

    def __init__(self, in_channels=128, num_classes=3):
        super().__init__()
        self.cls_tower = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
        )
        self.cls_head  = nn.Conv2d(64, num_classes, 1)   # 클래스 분류
        self.reg_head  = nn.Conv2d(64, 4, 1)             # bbox 회귀 (l,t,r,b)
        self.ctr_head  = nn.Conv2d(64, 1, 1)             # centerness

    def forward(self, feat):
        feat = self.cls_tower(feat)
        cls  = self.cls_head(feat)   # (B, C, H, W)
        reg  = self.reg_head(feat).exp()  # 양수 값으로
        ctr  = self.ctr_head(feat)
        return cls, reg, ctr


class DepthHead(nn.Module):
    """U-Net style depth estimation head — 원본 해상도로 upsample"""

    def __init__(self, in_channels=128):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid()   # 0~1 사이로 정규화 (실제 depth = x * 100m)
        )

    def forward(self, feat):
        return self.decoder(feat)


class SegmentationHead(nn.Module):
    """DeepLab-lite style segmentation head"""

    def __init__(self, in_channels=128, num_classes=NUM_CLASSES):
        super().__init__()
        # ASPP-lite: 다중 dilation conv
        self.aspp1 = nn.Conv2d(in_channels, 64, 1)
        self.aspp2 = nn.Conv2d(in_channels, 64, 3, padding=6,  dilation=6)
        self.aspp3 = nn.Conv2d(in_channels, 64, 3, padding=12, dilation=12)
        self.proj  = nn.Conv2d(192, 128, 1)

        self.decoder = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, num_classes, 1),
        )

    def forward(self, feat):
        a1 = F.relu(self.aspp1(feat))
        a2 = F.relu(self.aspp2(feat))
        a3 = F.relu(self.aspp3(feat))
        feat = self.proj(torch.cat([a1, a2, a3], dim=1))
        return self.decoder(feat)


class MultiTaskModel(nn.Module):
    """
    공유 ResNet-18 backbone + FPN + 3개 태스크 헤드
    """

    def __init__(self, pretrained=True):
        super().__init__()

        # Shared backbone
        resnet = models.resnet18(weights='DEFAULT' if pretrained else None)
        self.stem   = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1   # 64ch
        self.layer2 = resnet.layer2   # 128ch
        self.layer3 = resnet.layer3   # 256ch
        self.layer4 = resnet.layer4   # 512ch

        # Neck
        self.fpn = FPNNeck(in_channels=(64,128,256,512), out_channels=128)

        # Task heads
        self.det_head   = DetectionHead(in_channels=128, num_classes=3)
        self.depth_head = DepthHead(in_channels=128)
        self.seg_head   = SegmentationHead(in_channels=128, num_classes=NUM_CLASSES)

    def forward(self, x):
        # Backbone
        x  = self.stem(x)
        f1 = self.layer1(x)    # stride 4
        f2 = self.layer2(f1)   # stride 8
        f3 = self.layer3(f2)   # stride 16
        f4 = self.layer4(f3)   # stride 32

        # FPN
        p1, p2, p3, p4 = self.fpn([f1, f2, f3, f4])

        # Task heads — 각 태스크에 적합한 스케일 사용
        det_cls, det_reg, det_ctr = self.det_head(p3)    # stride 16, 중간 스케일
        depth_pred = self.depth_head(p4)                  # stride 32 → upsample
        seg_pred   = self.seg_head(p3)                    # stride 16 → upsample

        return det_cls, det_reg, det_ctr, depth_pred, seg_pred


# 모델 생성 + shape 확인
model = MultiTaskModel(pretrained=True).to(device)

dummy = torch.randn(2, 3, IMG_H, IMG_W).to(device)
with torch.no_grad():
    det_cls, det_reg, det_ctr, depth_pred, seg_pred = model(dummy)

print(f'Det cls:    {det_cls.shape}')     # (2, 3, 16, 16)
print(f'Det reg:    {det_reg.shape}')     # (2, 4, 16, 16)
print(f'Depth:      {depth_pred.shape}')  # (2, 1, 256, 256)
print(f'Seg:        {seg_pred.shape}')    # (2, 5, 128, 128)

total_params = sum(p.numel() for p in model.parameters())
print(f'파라미터: {total_params:,} ({total_params/1e6:.1f}M)')

## 3. GradNorm 손실 균형 구현

In [ ]:
class GradNormLoss(nn.Module):
    """
    GradNorm (Chen et al., ICML 2018)

    태스크별 loss 가중치 w_i를 자동 조정:
    - 각 태스크의 gradient norm G_i 측정
    - 목표 norm: G_bar * r_i^alpha (r_i = 상대 학습 속도)
    - GradNorm loss = Σ |G_i - G_tilde_i|

    alpha: 균형 강도 (크면 빠른 태스크 억제 강도 높음)
    """

    def __init__(self, num_tasks=3, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        # 학습 가능한 가중치 (log-space로 양수 보장)
        self.log_weights = nn.Parameter(torch.zeros(num_tasks))
        self.initial_losses = None   # 초기 손실 (L_i(0) for r_i 계산)

    @property
    def weights(self):
        return self.log_weights.exp()

    def forward(self, losses):
        """
        losses: list of task losses (tensor)
        Returns: weighted sum of losses
        """
        if self.initial_losses is None:
            self.initial_losses = [l.detach() for l in losses]

        weights = self.weights
        weighted_losses = [w * l for w, l in zip(weights, losses)]
        total_loss = sum(weighted_losses)
        return total_loss, weights.detach()

    def compute_gradnorm_loss(self, losses, shared_params):
        """
        GradNorm 보조 손실 계산 (가중치 업데이트용)
        shared_params: backbone의 마지막 레이어 파라미터 (gradient norm 기준점)
        """
        weights = self.weights

        # 각 태스크의 gradient norm
        G = []
        for i, loss in enumerate(losses):
            grads = torch.autograd.grad(
                weights[i] * loss, shared_params,
                create_graph=True, retain_graph=True, allow_unused=True
            )
            grad_norm = sum(
                g.norm() for g in grads if g is not None
            )
            G.append(grad_norm)

        G_bar = sum(G) / len(G)  # 평균 gradient norm

        # 상대 학습 속도 r_i = (L_i / L_i(0)) / 평균
        if self.initial_losses is not None:
            r = [l.detach() / (l0 + 1e-8)
                 for l, l0 in zip(losses, self.initial_losses)]
            r_mean = sum(r) / len(r)
            r_hat = [ri / (r_mean + 1e-8) for ri in r]
        else:
            r_hat = [1.0] * len(losses)

        # 목표 gradient norm: G_bar * r_hat_i^alpha
        G_tilde = [G_bar.detach() * (rh ** self.alpha) for rh in r_hat]

        # GradNorm loss
        L_grad = sum(abs(g - gt) for g, gt in zip(G, G_tilde))
        return L_grad


def compute_task_losses(det_cls, det_reg, det_ctr, depth_pred, seg_pred,
                        depth_gt, seg_gt, img_h=IMG_H, img_w=IMG_W):
    """3개 태스크 개별 손실 계산"""

    # Detection loss (간소화: focal loss 대신 BCE + smooth L1)
    # 합성 데이터에서 GT 마스크 생성
    B, C, H, W = det_cls.shape
    cls_target = torch.zeros_like(det_cls)
    l_det = F.binary_cross_entropy_with_logits(det_cls, cls_target)

    # Depth loss (SILog — Scale-Invariant Log loss)
    depth_pred_full = F.interpolate(depth_pred, size=(img_h, img_w),
                                    mode='bilinear', align_corners=True)
    depth_pred_m = depth_pred_full * 100.0  # 0~100m 스케일 복원
    log_diff = torch.log(depth_pred_m + 1e-6) - torch.log(depth_gt + 1e-6)
    l_depth = log_diff.pow(2).mean() - 0.5 * log_diff.mean().pow(2)

    # Segmentation loss (CrossEntropy)
    seg_pred_up = F.interpolate(seg_pred, size=(img_h, img_w),
                                mode='bilinear', align_corners=True)
    l_seg = F.cross_entropy(seg_pred_up, seg_gt)

    return l_det, l_depth, l_seg


print('손실 함수 + GradNorm 준비 완료')

## 4. 학습 파이프라인 (MTL vs Single-Task 비교)

In [ ]:
def train_mtl(model, train_loader, val_loader, epochs=20, use_gradnorm=True):
    """MTL 학습 (GradNorm 유무 선택 가능)"""

    gradnorm = GradNormLoss(num_tasks=3, alpha=0.5).to(device)
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(gradnorm.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # 공유 backbone 마지막 레이어 파라미터 (GradNorm 기준점)
    shared_params = list(model.layer4.parameters())

    history = {'det':[], 'depth':[], 'seg':[],
               'w_det':[], 'w_depth':[], 'w_seg':[]}

    for epoch in range(1, epochs+1):
        model.train()
        epoch_losses = {'det':0,'depth':0,'seg':0}
        epoch_weights = {'det':0,'depth':0,'seg':0}
        n = 0

        for imgs, depths, segs, _ in train_loader:
            imgs   = imgs.to(device)
            depths = depths.to(device)
            segs   = segs.to(device)

            det_cls, det_reg, det_ctr, depth_pred, seg_pred = model(imgs)
            l_det, l_depth, l_seg = compute_task_losses(
                det_cls, det_reg, det_ctr, depth_pred, seg_pred, depths, segs
            )

            if use_gradnorm:
                total_loss, weights = gradnorm([l_det, l_depth, l_seg])

                # GradNorm 보조 손실로 가중치 업데이트
                optimizer.zero_grad()
                total_loss.backward(retain_graph=True)

                # GradNorm loss는 log_weights에만 적용
                for p in model.parameters():
                    p.grad = None
                l_gn = gradnorm.compute_gradnorm_loss([l_det, l_depth, l_seg], shared_params)
                l_gn.backward()

                epoch_weights['det']   += weights[0].item()
                epoch_weights['depth'] += weights[1].item()
                epoch_weights['seg']   += weights[2].item()
            else:
                # 단순 합산 (equal weighting)
                total_loss = l_det + l_depth + l_seg
                optimizer.zero_grad()
                total_loss.backward()
                epoch_weights['det']   += 1.0
                epoch_weights['depth'] += 1.0
                epoch_weights['seg']   += 1.0

            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            epoch_losses['det']   += l_det.item()
            epoch_losses['depth'] += l_depth.item()
            epoch_losses['seg']   += l_seg.item()
            n += 1

        scheduler.step()

        for k in epoch_losses:
            history[k].append(epoch_losses[k]/n)
        for k, wk in zip(['w_det','w_depth','w_seg'], ['det','depth','seg']):
            history[k].append(epoch_weights[wk]/n)

        if epoch % 5 == 0 or epoch == 1:
            w = history
            print(f'Epoch {epoch:2d}/{epochs} | '
                  f'Det={w["det"][-1]:.4f} '
                  f'Depth={w["depth"][-1]:.4f} '
                  f'Seg={w["seg"][-1]:.4f} | '
                  f'W=[{w["w_det"][-1]:.2f},{w["w_depth"][-1]:.2f},{w["w_seg"][-1]:.2f}]')

    return history


# ── GradNorm 사용 MTL 학습 ────────────────────
print('=== MTL with GradNorm ===')
model_gn = MultiTaskModel(pretrained=True).to(device)
hist_gn = train_mtl(model_gn, train_loader, val_loader, epochs=20, use_gradnorm=True)

print()
print('=== MTL without GradNorm (Equal weights) ===')
model_eq = MultiTaskModel(pretrained=True).to(device)
hist_eq = train_mtl(model_eq, train_loader, val_loader, epochs=20, use_gradnorm=False)

In [ ]:
# ── 결과 시각화 ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

tasks = ['det', 'depth', 'seg']
task_names = ['Detection', 'Depth', 'Segmentation']
colors_gn = ['steelblue', 'coral', 'green']

# 상단: 태스크별 loss 비교
for i, (task, name, color) in enumerate(zip(tasks, task_names, colors_gn)):
    ax = axes[0, i]
    ax.plot(hist_gn[task], label='GradNorm', color=color, linewidth=2)
    ax.plot(hist_eq[task], label='Equal weights', color=color, linewidth=2, linestyle='--', alpha=0.6)
    ax.set_title(f'{name} Loss')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

# 하단: GradNorm 가중치 변화
ax = axes[1, 0]
ax.plot(hist_gn['w_det'],   label='Det',   color=colors_gn[0])
ax.plot(hist_gn['w_depth'], label='Depth', color=colors_gn[1])
ax.plot(hist_gn['w_seg'],   label='Seg',   color=colors_gn[2])
ax.set_title('GradNorm 가중치 변화 (w_i)')
ax.set_xlabel('Epoch')
ax.legend()
ax.grid(True, alpha=0.3)

# 최종 loss 비교 막대
ax = axes[1, 1]
x = np.arange(3)
final_gn = [hist_gn[t][-1] for t in tasks]
final_eq = [hist_eq[t][-1] for t in tasks]

bars1 = ax.bar(x - 0.2, final_gn, 0.4, label='GradNorm', color=colors_gn)
bars2 = ax.bar(x + 0.2, final_eq, 0.4, label='Equal', color=colors_gn, alpha=0.5, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(task_names)
ax.set_title('최종 Loss 비교 (GradNorm vs Equal)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# GradNorm 개선율
ax = axes[1, 2]
improve = [(eq-gn)/(eq+1e-8)*100 for gn, eq in zip(final_gn, final_eq)]
bars = ax.bar(task_names, improve, color=colors_gn)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('GradNorm 손실 개선율 (%)')
ax.set_ylabel('개선율 (%)')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, improve):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val:+.1f}%', ha='center', fontsize=10)

plt.suptitle('Multi-Task Learning: GradNorm vs Equal Weighting', fontsize=13)
plt.tight_layout()
plt.savefig('mtl_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 정성적 결과: 3개 태스크 동시 추론 ──────────
model_gn.eval()
imgs, depths_gt, segs_gt, _ = next(iter(val_loader))

with torch.no_grad():
    det_cls, det_reg, det_ctr, depth_pred, seg_pred = model_gn(imgs.to(device))

denorm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for col in range(4):
    img_show = denorm(imgs[col]).permute(1,2,0).clamp(0,1).numpy()
    axes[0, col].imshow(img_show)
    axes[0, col].set_title(f'입력 이미지 {col+1}')

    depth_show = F.interpolate(depth_pred, size=(IMG_H, IMG_W),
                               mode='bilinear', align_corners=True)
    dp = depth_show[col, 0].cpu().numpy() * 100
    axes[1, col].imshow(dp, cmap='plasma', vmin=0, vmax=50)
    axes[1, col].set_title('예측 Depth')

    seg_up = F.interpolate(seg_pred, size=(IMG_H, IMG_W),
                           mode='bilinear', align_corners=True)
    seg_show = seg_up[col].argmax(0).cpu().numpy()
    color_seg = np.zeros((*seg_show.shape, 3), dtype=np.uint8)
    for cls_id, color in CLASS_COLORS.items():
        color_seg[seg_show == cls_id] = color
    axes[2, col].imshow(color_seg)
    axes[2, col].set_title('예측 Segmentation')

row_labels = ['RGB 입력', 'Depth 예측', 'Seg 예측']
for ax, label in zip(axes[:, 0], row_labels):
    ax.set_ylabel(label, fontsize=10)

for ax in axes.flat:
    ax.axis('off')
axes[0, 0].axis('on'); axes[0, 0].set_xticks([]); axes[0, 0].set_yticks([])

plt.suptitle('MTL 모델 — 단일 추론으로 3개 태스크 동시 출력', fontsize=13)
plt.tight_layout()
plt.savefig('mtl_qualitative.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. 면접 Q&A

**Q. Multi-Task Learning에서 negative transfer란 무엇인가요?**

> MTL에서 한 태스크의 학습이 다른 태스크 성능을 **저하**시키는 현상입니다. 예를 들어 Detection과 Depth는 공간적 위치 정보를 공유하므로 positive transfer가 발생하지만, Detection(객체 경계 강조)과 Depth(부드러운 공간 표현)는 최적 표현 방식이 달라 충돌할 수 있습니다. Single-task 대비 MTL 성능이 떨어지면 negative transfer로 진단합니다.

**Q. GradNorm은 어떻게 학습 균형을 맞추나요?**

> 학습 중 각 태스크의 gradient norm G_i를 측정합니다. 어떤 태스크가 너무 빠르게 학습되면(G_i > G_bar) 그 가중치 w_i를 줄이고, 너무 느리면 늘립니다. 가중치는 별도의 GradNorm loss `L = Σ|G_i - G_tilde_i|`로 업데이트되며, 이는 task loss와 독립적으로 w_i 파라미터에만 역전파합니다.

**Q. FPN(Feature Pyramid Network)을 MTL에서 사용하는 이유는?**

> 태스크마다 최적 해상도가 다릅니다. Segmentation은 픽셀 단위 정확도가 필요해 고해상도 feature가 유리하고, Detection은 중간 해상도에서 객체 수준 표현이 좋습니다. FPN은 top-down 경로로 고수준 의미 정보를 저수준 feature에 전달해 모든 스케일에서 풍부한 표현을 만듭니다.

**Q. 자율주행에서 MTL의 실용적 장점은?**

> 1. **추론 속도**: 1개 backbone으로 3개 태스크를 동시 처리 → 지연 시간 단축
> 2. **공유 표현**: Det이 잘 학습하면 같은 backbone의 Depth/Seg도 개선
> 3. **메모리 효율**: 3개 별도 모델 대비 메모리 1/3 (backbone 공유)
> Tesla FSD, Waymo 등 실제 자율주행 시스템에서 MTL 기반 unified perception을 사용합니다.

In [ ]:
# ── 최종 결과 요약 ──────────────────────────────
final_gn = {t: hist_gn[t][-1] for t in ['det','depth','seg']}
final_eq = {t: hist_eq[t][-1] for t in ['det','depth','seg']}
improve  = {t: (final_eq[t]-final_gn[t])/(final_eq[t]+1e-8)*100
            for t in ['det','depth','seg']}

print('=' * 65)
print('skillup_round6 / 03 Multi-Task Learning — 최종 결과')
print('=' * 65)
print(f'  모델:   ResNet-18 + FPN + Det/Depth/Seg Head (3개 태스크)')
print(f'  데이터: 합성 도시 장면 (train={len(train_ds)}, val={len(val_ds)})')
print()
print(f'  {'태스크':<12} {'GradNorm':<12} {'Equal':<12} {'개선률'}')
print(f'  {'-'*48}')
for t, name in zip(['det','depth','seg'], ['Detection','Depth','Segmentation']):
    print(f'  {name:<12} {final_gn[t]:<12.4f} {final_eq[t]:<12.4f} {improve[t]:+.1f}%')
print()
print('구현 핵심:')
print('  1. 공유 ResNet-18 backbone + FPN neck (다중 스케일)')
print('  2. 3개 태스크 헤드: FCOS Det / U-Net Depth / DeepLab-lite Seg')
print('  3. GradNorm: gradient norm 기반 자동 가중치 조정')
print('  4. GradNorm vs Equal weighting 비교 (positive/negative transfer 분석)')
print('=' * 65)